# Indigo AI Personalization Project

## Step 4 Analytics Notebook

This notebook supports the analytics module for evaluating whether Indigo should invest in an AI-driven lifestyle personalization engine.

### Scope of this section
This notebook begins with simulated customer-level data for 10,000 Plum Rewards members. The simulation focuses on two customer segments:

- **Book-only customers**
- **Multi-category customers**

The simulated dataset will be used in later sections for:
- CLV estimation
- scenario analysis
- sensitivity testing

## 1. Project Setup

In [17]:
import numpy as np
import pandas as pd

In [18]:
# Reproducibility
np.random.seed(42)

## 2. Assumptions and Segmet Definations

We simulate 10,000 customers across two segments:

- **Book-only customers**
- **Multi-category customers**

### Segment-level assumptions

| Segment | Purchase Frequency (orders/year) | Average Order Value ($) | Contribution Margin | Retention Probability |
|--------|-----------------------------------|--------------------------|---------------------|----------------------|
| Book-only | 3 | 25 | 30% | 60% |
| Multi-category | 5 | 45 | 35% | 80% |

### Distribution choices

- **Purchase frequency** is modeled using a **zero-truncated Poisson distribution**, so every simulated customer has at least one annual order.
- **Average order value (AOV)** is modeled using a **log-normal distribution** calibrated so the arithmetic mean matches the target segment AOV.
- **Retention** is modeled as a **Bernoulli variable**, representing whether a customer remains active in the next period.
- **Margin** is treated as fixed within segment for simplicity.

## 3. Simulating 10,000 Customers

In [19]:
# -----------------------------
# Simulation parameters
# -----------------------------
N = 10000

# Segment mix assumption
book_share = 0.70
multi_share = 0.30

# Book-only segment parameters
book_freq_lambda = 3
book_aov_mean = 25
book_aov_sigma = 0.30
book_margin = 0.30
book_retention_prob = 0.60

# Multi-category segment parameters
multi_freq_lambda = 5
multi_aov_mean = 45
multi_aov_sigma = 0.40
multi_margin = 0.35
multi_retention_prob = 0.80

In [20]:
# Generate segment labels based on the specified mix
segments = np.random.choice(
    ["book_only", "multi_category"],
    size=N,
    p=[book_share, multi_share]
)

df = pd.DataFrame({
    "customer_id": np.arange(1, N + 1),
    "segment": segments
})

df.head()

,customer_id,segment
0,1,book_only
1,2,multi_category
2,3,multi_category
3,4,book_only
4,5,book_only


In [21]:
# ------------------------------
# Simulate customer variables based on segment
# ------------------------------

def zero_truncated_poisson(lam, size):
    samples = np.random.poisson(lam=lam, size=size)
    while (samples == 0).any():
        zero_mask = samples == 0
        samples[zero_mask] = np.random.poisson(lam=lam, size=zero_mask.sum())
    return samples

def lognormal_mu_from_mean(target_mean, sigma):
    return np.log(target_mean) - 0.5 * sigma**2

# Initialize columns
df["purchase_frequency"] = np.nan
df["AOV"] = np.nan
df["margin"] = np.nan
df["retention"] = np.nan

# Masks
mask_book = df["segment"] == "book_only"
mask_multi = df["segment"] == "multi_category"

n_book = mask_book.sum()
n_multi = mask_multi.sum()

book_aov_mu = lognormal_mu_from_mean(book_aov_mean, book_aov_sigma)
multi_aov_mu = lognormal_mu_from_mean(multi_aov_mean, multi_aov_sigma)

# -----------------------------
# Book-only customers
# -----------------------------
df.loc[mask_book, "purchase_frequency"] = zero_truncated_poisson(
    lam=book_freq_lambda,
    size=n_book
)

df.loc[mask_book, "AOV"] = np.random.lognormal(
    mean=book_aov_mu,
    sigma=book_aov_sigma,
    size=n_book
)

df.loc[mask_book, "margin"] = book_margin

df.loc[mask_book, "retention"] = np.random.binomial(
    n=1,
    p=book_retention_prob,
    size=n_book
)

# -----------------------------
# Multi-category customers
# -----------------------------
df.loc[mask_multi, "purchase_frequency"] = zero_truncated_poisson(
    lam=multi_freq_lambda,
    size=n_multi
)

df.loc[mask_multi, "AOV"] = np.random.lognormal(
    mean=multi_aov_mu,
    sigma=multi_aov_sigma,
    size=n_multi
)

df.loc[mask_multi, "margin"] = multi_margin

df.loc[mask_multi, "retention"] = np.random.binomial(
    n=1,
    p=multi_retention_prob,
    size=n_multi
)

df.head()

,customer_id,segment,purchase_frequency,AOV,margin,retention
0,1,book_only,2.0,24.736906,0.30,0.0
1,2,multi_category,7.0,38.965100,0.35,0.0
2,3,multi_category,5.0,45.441286,0.35,1.0
3,4,book_only,3.0,31.115186,0.30,1.0
4,5,book_only,7.0,22.136022,0.30,0.0


In [22]:
# ---------------------
# Data cleaning / clipping
# ---------------------

# Apply light caps to avoid unrealistic outliers
df["purchase_frequency"] = df["purchase_frequency"].clip(upper=15)
df["AOV"] = df["AOV"].clip(lower=5, upper=200)

# Round for readability
df["purchase_frequency"] = df["purchase_frequency"].astype(int)
df["AOV"] = df["AOV"].round(2)
df["margin"] = df["margin"].round(2)
df["retention"] = df["retention"].astype(int)

df.head()

,customer_id,segment,purchase_frequency,AOV,margin,retention
0,1,book_only,2,24.74,0.30,0
1,2,multi_category,7,38.97,0.35,0
2,3,multi_category,5,45.44,0.35,1
3,4,book_only,3,31.12,0.30,1
4,5,book_only,7,22.14,0.30,0


In [23]:
assert df["customer_id"].is_unique
assert not df.isnull().any().any()
assert df["retention"].isin([0, 1]).all()

print("Dataset shape:", df.shape)
print("\nSegment counts:")
print(df["segment"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

print("\nSegment-level means:")
print(
    df.groupby("segment")[["purchase_frequency", "AOV", "retention"]]
    .mean()
    .round(2)
)

Dataset shape: (10000, 6)

Segment counts:
segment
book_only         7113
multi_category    2887
Name: count, dtype: int64

Missing values:
customer_id           0
segment               0
purchase_frequency    0
AOV                   0
margin                0
retention             0
dtype: int64

Segment-level means:
                purchase_frequency    AOV  retention
segment                                             
book_only                     3.17  24.82       0.59
multi_category                5.07  45.16       0.81


## 4. Summary Statistics

In [24]:
summary_stats = df.groupby("segment").agg(
    customers=("customer_id", "count"),
    avg_purchase_frequency=("purchase_frequency", "mean"),
    avg_AOV=("AOV", "mean"),
    avg_margin=("margin", "mean"),
    avg_retention=("retention", "mean")
).reset_index()

summary_stats["avg_purchase_frequency"] = summary_stats["avg_purchase_frequency"].round(2)
summary_stats["avg_AOV"] = summary_stats["avg_AOV"].round(2)
summary_stats["avg_margin"] = summary_stats["avg_margin"].round(2)
summary_stats["avg_retention"] = summary_stats["avg_retention"].round(2)

summary_stats

,segment,customers,avg_purchase_frequency,avg_AOV,avg_margin,avg_retention
0,book_only,7113,3.17,24.82,0.30,0.59
1,multi_category,2887,5.07,45.16,0.35,0.81


In [25]:
df["annual_profit"] = (
    df["purchase_frequency"] * df["AOV"] * df["margin"]
).round(2)

df.head()

,customer_id,segment,purchase_frequency,AOV,margin,retention,annual_profit
0,1,book_only,2,24.74,0.30,0,14.84
1,2,multi_category,7,38.97,0.35,0,95.48
2,3,multi_category,5,45.44,0.35,1,79.52
3,4,book_only,3,31.12,0.30,1,28.01
4,5,book_only,7,22.14,0.30,0,46.49


In [26]:
# Summary including annual profit

profit_summary = df.groupby("segment").agg(
    customers=("customer_id", "count"),
    avg_purchase_frequency=("purchase_frequency", "mean"),
    avg_AOV=("AOV", "mean"),
    avg_margin=("margin", "mean"),
    avg_retention=("retention", "mean"),
    avg_annual_profit=("annual_profit", "mean")
).reset_index()

profit_summary = profit_summary.round({
    "avg_purchase_frequency": 2,
    "avg_AOV": 2,
    "avg_margin": 2,
    "avg_retention": 2,
    "avg_annual_profit": 2
})

profit_summary

,segment,customers,avg_purchase_frequency,avg_AOV,avg_margin,avg_retention,avg_annual_profit
0,book_only,7113,3.17,24.82,0.30,0.59,23.69
1,multi_category,2887,5.07,45.16,0.35,0.81,80.33


## 5. Export Simulated Dataset

In [27]:
output_path = "indigo_simulated_customers.csv"
df.to_csv(output_path, index=False)
print(f"Simulated dataset exported to: {output_path}")

Simulated dataset exported to: indigo_simulated_customers.csv


## 6. CLV Modeling

## 7. Scenario Analysis

## 8. Sensitivity Analysis